Professor: Yunlong Wang
Author: Stephen Handy
CSC381-01
Fall 2026

Report_2: US Flight Delays & Cancellations 2025-2026

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

csv_path = Path("dataset/flights_sample.csv")

df = pd.read_csv(csv_path)
print(df.shape)
df.head()


In [ ]:
df = df[
    (df["cancelled"] == 0)
    & (df["diverted"] == 0)
    & (df["arr_delay"].notna())
].copy()


In [ ]:
df["sched_dep_hour"] = df["sched_dep_time"] // 100


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

feature_columns = [
    "airline_code", "origin", "dest", "month", "day_of_week", "sched_dep_hour", "dep_time_block", "distance", "sched_elapsed_time"
]

categorical = [
    "airline_code", "origin", "dest", "month", "day_of_week", "dep_time_block"
]

numeric = ["sched_dep_hour", "distance", "sched_elapsed_time"]

preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("numeric", "passthrough", numeric)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=1.0))
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)


In [ ]:
df["flight_date"] = pd.to_datetime(df["flight_date"])
df = df.sort_values("flight_date")

split_date = df["flight_date"].quantile(0.80)

train = df[df["flight_date"] <= split_date].copy()
test = df[df["flight_date"] > split_date].copy()

X_train = train[feature_columns]
X_train = train["arr_delay"]

X_test = test[feature_columns]
X_test = test["arr_delay"]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
